# Review-period and lead-time sensitivity

This notebook reuses the saved daily forecasts from the main workflow; it does not refit any forecasting model. Run the cells in order. Set `M5_DATA_DIR` as described in the main notebook.


## Software environment


In [ ]:
import importlib.metadata as metadata
import json
import os
import platform
import sys

PACKAGE_NAMES = [
    "numpy",
    "pandas",
    "scipy",
    "statsmodels",
    "pmdarima",
    "scikit-learn",
    "xgboost",
    "torch",
    "tirex2",
    "matplotlib",
]

software_environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
}

for package_name in PACKAGE_NAMES:
    try:
        software_environment[package_name] = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        software_environment[package_name] = "not installed"

print(json.dumps(software_environment, indent=2))


## 1. Reconstruct policy targets and run six timing policies

This main cell recovers the demand classes, uses the price-based active period, constructs 14-, 21- and 28-day protection targets, and simulates all compatible review-period and lead-time combinations.


In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# 1. Paths and settings
# ============================================================

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))
OUTPUT_DIR = DATA_DIR / "chapter5_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_FILE = DATA_DIR / "selected_sample_ids.csv"
SALES_FILE = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_FILE = DATA_DIR / "calendar.csv"
PRICES_FILE = DATA_DIR / "sell_prices.csv"

WEEKLY_FORECAST_FILE = (
    OUTPUT_DIR / "inventory_weekly_all_model_forecasts.csv"
)

MODELS = [
    "seasonal_naive",
    "ets",
    "arima_sarima",
    "xgboost",
    "tirex2"
]

QUANTILE_LEVELS = [0.90, 0.95, 0.99]

# Review period R and lead time L, measured in days
POLICIES = [
    (7, 7),
    (7, 14),
    (7, 21),      # Original baseline
    (14, 7),
    (14, 14),
    (21, 7)
]

WEEKLY_ORIGINS = list(range(1829, 1935, 7))

# Six calibration origins plus three validation origins.
# The final-test origin d_1913 remains excluded from the
# residual pool.
ERROR_ORIGINS = [
    1661,
    1689,
    1717,
    1745,
    1773,
    1801,
    1829,
    1857,
    1885
]

SIMULATION_DAYS = np.arange(1830, 1942)

# d_1830--d_1913: warm-up
# d_1914--d_1941: final test
FINAL_TEST_START = 1914

ANNUAL_HOLDING_RATE = 0.01
LOST_SALES_RATE = 0.125
FIXED_ORDERING_COST = 0.50
MINIMUM_FILL_RATE = 0.95

assert all(
    review_period + lead_time <= 28
    for review_period, lead_time in POLICIES
)


# ============================================================
# 2. Load the selected sample and M5 demand
# ============================================================

sample = pd.read_csv(SAMPLE_FILE)
sample["id"] = sample["id"].astype(str)

assert sample["id"].nunique() == 500


# ------------------------------------------------------------
# Recover the demand classes if they are not in the sample file
# ------------------------------------------------------------

if (
    "demand_class" not in sample.columns
    and "demand_group" in sample.columns
):
    sample = sample.rename(
        columns={"demand_group": "demand_class"}
    )


if "demand_class" not in sample.columns:

    classification_sources = [
        OUTPUT_DIR
        / "inventory_weekly_order_up_to_targets.csv",

        OUTPUT_DIR
        / "inventory_weekly_all_model_forecasts.csv",

        OUTPUT_DIR
        / "seasonal_naive_forecasts.csv"
    ]

    classification_lookup = None

    for classification_file in classification_sources:

        if not classification_file.exists():
            continue

        available_columns = pd.read_csv(
            classification_file,
            nrows=0
        ).columns.tolist()

        if "demand_class" in available_columns:

            candidate_lookup = (
                pd.read_csv(
                    classification_file,
                    usecols=[
                        "id",
                        "demand_class"
                    ]
                )
                .dropna(subset=["demand_class"])
                .drop_duplicates()
            )

        elif "demand_group" in available_columns:

            candidate_lookup = (
                pd.read_csv(
                    classification_file,
                    usecols=[
                        "id",
                        "demand_group"
                    ]
                )
                .rename(
                    columns={
                        "demand_group": "demand_class"
                    }
                )
                .dropna(subset=["demand_class"])
                .drop_duplicates()
            )

        else:
            continue

        candidate_lookup["id"] = (
            candidate_lookup["id"].astype(str)
        )

        # Check that each ID has only one classification
        inconsistent_ids = (
            candidate_lookup
            .groupby("id")["demand_class"]
            .nunique()
            .gt(1)
        )

        if inconsistent_ids.any():
            raise ValueError(
                f"Inconsistent demand classes in "
                f"{classification_file.name}."
            )

        candidate_lookup = (
            candidate_lookup
            .drop_duplicates("id")
        )

        matched_ids = sample["id"].isin(
            candidate_lookup["id"]
        ).sum()

        if matched_ids == 500:
            classification_lookup = candidate_lookup
            print(
                "Demand classes recovered from:",
                classification_file.name
            )
            break


    if classification_lookup is None:
        raise FileNotFoundError(
            "The demand classes could not be recovered "
            "from the existing result files."
        )


    sample = sample.merge(
        classification_lookup,
        on="id",
        how="left",
        validate="one_to_one"
    )


if sample["demand_class"].isna().any():
    raise ValueError(
        "At least one selected series has no demand class."
    )


# Standardise the spelling
sample["demand_class"] = (
    sample["demand_class"]
    .astype(str)
    .str.strip()
    .str.capitalize()
)


# Check the balanced sample
class_counts = (
    sample["demand_class"]
    .value_counts()
    .sort_index()
)

expected_class_counts = pd.Series(
    {
        "Erratic": 125,
        "Intermittent": 125,
        "Lumpy": 125,
        "Smooth": 125
    }
).sort_index()

if not class_counts.equals(expected_class_counts):
    raise AssertionError(
        "The recovered sample is not balanced:\n"
        f"{class_counts}"
    )

print("\nDemand classes:")
print(class_counts)


# ------------------------------------------------------------
# Load the M5 demand observations
# ------------------------------------------------------------

sales = pd.read_csv(SALES_FILE)
sales["id"] = sales["id"].astype(str)

selected_sales = sample[["id"]].merge(
    sales,
    on="id",
    validate="one_to_one"
)

day_columns = sorted(
    [
        column
        for column in sales.columns
        if column.startswith("d_")
    ],
    key=lambda column: int(
        column.split("_")[1]
    )
)

demand_matrix = selected_sales[
    day_columns
].to_numpy(dtype=float)

series_ids = sample["id"].tolist()
number_of_series = len(series_ids)


# ------------------------------------------------------------
# Recover the first active day if necessary
# ------------------------------------------------------------

# ------------------------------------------------------------
# Define the active period using the first valid price week
# ------------------------------------------------------------

calendar_for_active_period = pd.read_csv(
    CALENDAR_FILE
)

prices_for_active_period = pd.read_csv(
    PRICES_FILE
)

calendar_for_active_period["day_number"] = (
    calendar_for_active_period["d"]
    .str.replace(
        "d_",
        "",
        regex=False
    )
    .astype(int)
)

first_day_by_week = (
    calendar_for_active_period
    .groupby("wm_yr_wk")["day_number"]
    .min()
)

valid_prices = prices_for_active_period[
    np.isfinite(
        prices_for_active_period["sell_price"]
    )
    & (
        prices_for_active_period["sell_price"] > 0
    )
].copy()

first_price_week = (
    valid_prices
    .groupby(
        [
            "store_id",
            "item_id"
        ],
        as_index=False
    )["wm_yr_wk"]
    .min()
    .rename(
        columns={
            "wm_yr_wk": "first_price_week"
        }
    )
)

first_price_week["first_active_day"] = (
    first_price_week["first_price_week"]
    .map(first_day_by_week)
)

active_period_lookup = (
    selected_sales[
        [
            "id",
            "store_id",
            "item_id"
        ]
    ]
    .merge(
        first_price_week[
            [
                "store_id",
                "item_id",
                "first_price_week",
                "first_active_day"
            ]
        ],
        on=[
            "store_id",
            "item_id"
        ],
        how="left",
        validate="one_to_one"
    )
)

if active_period_lookup[
    "first_active_day"
].isna().any():

    missing_ids = active_period_lookup.loc[
        active_period_lookup[
            "first_active_day"
        ].isna(),
        "id"
    ].tolist()

    raise ValueError(
        "No valid positive price was found for "
        f"these selected series: {missing_ids[:5]}"
    )

first_active_day = (
    active_period_lookup
    .set_index("id")["first_active_day"]
    .reindex(series_ids)
    .astype(int)
    .to_numpy()
)

sample["first_active_day"] = (
    first_active_day
)

print(
    "Series whose price-based active period "
    "begins after d_1:",
    (first_active_day > 1).sum()
)


print("\nSelected series:", number_of_series)
print(
    "Observed demand ends at:",
    day_columns[-1]
)
# ============================================================
# 3. Load the previously saved forecasts
# ============================================================

def read_forecasts(file_path, model_name=None):

    forecast_data = pd.read_csv(file_path)

    if model_name is not None:
        forecast_data["model"] = model_name

    required_columns = [
        "model",
        "id",
        "forecast_origin",
        "horizon",
        "target_day",
        "forecast"
    ]

    missing_columns = (
        set(required_columns)
        - set(forecast_data.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{file_path.name} is missing "
            f"{sorted(missing_columns)}"
        )

    forecast_data["id"] = (
        forecast_data["id"].astype(str)
    )

    forecast_data["model"] = (
        forecast_data["model"]
        .astype(str)
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .replace(
            {
                "tirex_2": "tirex2",
                "seasonal_naïve": "seasonal_naive"
            }
        )
    )

    integer_columns = [
        "forecast_origin",
        "horizon",
        "target_day"
    ]

    forecast_data[integer_columns] = (
        forecast_data[integer_columns].astype(int)
    )

    forecast_data["forecast"] = (
        forecast_data["forecast"].astype(float)
    )

    return forecast_data[required_columns]


forecast_parts = []

classical_files = {
    "seasonal_naive": "seasonal_naive_forecasts.csv",
    "ets": "ets_forecasts.csv",
    "arima_sarima": "arima_sarima_forecasts.csv",
    "xgboost": "xgboost_forecasts.csv"
}

for model_name, file_name in classical_files.items():

    file_path = OUTPUT_DIR / file_name

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    forecast_parts.append(
        read_forecasts(
            file_path=file_path,
            model_name=model_name
        )
    )


tirex_files = sorted(
    OUTPUT_DIR.glob(
        "tirex2_central_forecasts_origin_*.csv"
    )
)

if not tirex_files:
    raise FileNotFoundError(
        "The TiRex-2 central forecast files were not found."
    )

for file_path in tirex_files:
    forecast_parts.append(
        read_forecasts(
            file_path=file_path,
            model_name="tirex2"
        )
    )


if not WEEKLY_FORECAST_FILE.exists():
    raise FileNotFoundError(WEEKLY_FORECAST_FILE)

forecast_parts.append(
    read_forecasts(WEEKLY_FORECAST_FILE)
)


forecasts = pd.concat(
    forecast_parts,
    ignore_index=True
)

# The combined weekly file overlaps with some of the original
# forecast files. Keep one forecast for every unique key.
forecasts = (
    forecasts
    .sort_values(
        [
            "model",
            "id",
            "forecast_origin",
            "horizon"
        ]
    )
    .drop_duplicates(
        [
            "model",
            "id",
            "forecast_origin",
            "horizon"
        ],
        keep="last"
    )
    .merge(
        sample[["id", "demand_class"]],
        on="id",
        validate="many_to_one"
    )
)

required_origins = sorted(
    set(ERROR_ORIGINS + WEEKLY_ORIGINS)
)

forecasts = forecasts[
    forecasts["forecast_origin"].isin(required_origins)
].copy()


assert set(forecasts["model"].unique()) == set(MODELS)

assert (
    forecasts
    .groupby(
        [
            "model",
            "id",
            "forecast_origin"
        ]
    )
    .size()
    .eq(28)
    .all()
)

assert (
    forecasts
    .groupby("model")["forecast_origin"]
    .nunique()
    .eq(len(required_origins))
    .all()
)

assert np.isfinite(forecasts["forecast"]).all()
assert (forecasts["forecast"] >= 0).all()

print("Forecast rows loaded:", len(forecasts))


# ============================================================
# 4. Construct the daily price matrix
# ============================================================

calendar = pd.read_csv(CALENDAR_FILE)

calendar["target_day"] = (
    calendar["d"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)

prices = pd.read_csv(PRICES_FILE)

price_grid = pd.MultiIndex.from_product(
    [
        series_ids,
        SIMULATION_DAYS
    ],
    names=[
        "id",
        "target_day"
    ]
).to_frame(index=False)

price_grid = (
    price_grid
    .merge(
        selected_sales[
            [
                "id",
                "item_id",
                "store_id"
            ]
        ],
        on="id"
    )
    .merge(
        calendar[
            [
                "target_day",
                "wm_yr_wk"
            ]
        ],
        on="target_day"
    )
    .merge(
        prices[
            [
                "store_id",
                "item_id",
                "wm_yr_wk",
                "sell_price"
            ]
        ],
        on=[
            "store_id",
            "item_id",
            "wm_yr_wk"
        ],
        how="left"
    )
    .sort_values(
        [
            "id",
            "target_day"
        ]
    )
)

price_grid["sell_price"] = (
    price_grid
    .groupby("id")["sell_price"]
    .ffill()
)

assert price_grid["sell_price"].notna().all()

price_matrix = (
    price_grid
    .pivot(
        index="id",
        columns="target_day",
        values="sell_price"
    )
    .reindex(
        index=series_ids,
        columns=SIMULATION_DAYS
    )
    .to_numpy(dtype=float)
)


# ============================================================
# 5. Protection-period scales and forecast errors
# ============================================================

def calculate_protection_scales(
    protection_period,
    forecast_origins
):

    cumulative_demand = np.pad(
        np.cumsum(
            demand_matrix,
            axis=1
        ),
        (
            (0, 0),
            (1, 0)
        )
    )

    rolling_totals = (
        cumulative_demand[:, protection_period:]
        - cumulative_demand[:, :-protection_period]
    )

    scale_parts = []

    for forecast_origin in sorted(
        set(forecast_origins)
    ):

        # The last rolling window must be completely observed
        # at the forecast origin.
        history_end = (
            forecast_origin
            - protection_period
            + 1
        )

        scale_values = np.empty(number_of_series)

        for series_index in range(number_of_series):

            history = rolling_totals[
                series_index,
                first_active_day[series_index] - 1:
                history_end
            ]

            if len(history) > 1:
                standard_deviation = np.std(
                    history,
                    ddof=1
                )
            else:
                standard_deviation = 0.0

            if not np.isfinite(standard_deviation):
                standard_deviation = 0.0

            scale_values[series_index] = max(
                1.0,
                standard_deviation
            )

        scale_parts.append(
            pd.DataFrame(
                {
                    "id": series_ids,
                    "forecast_origin": forecast_origin,
                    "protection_scale": scale_values
                }
            )
        )

    return pd.concat(
        scale_parts,
        ignore_index=True
    )


def construct_actual_totals(protection_period):

    actual_parts = []

    for forecast_origin in ERROR_ORIGINS:

        actual_total = demand_matrix[
            :,
            forecast_origin:
            forecast_origin + protection_period
        ].sum(axis=1)

        actual_parts.append(
            pd.DataFrame(
                {
                    "id": series_ids,
                    "forecast_origin": forecast_origin,
                    "actual_total": actual_total
                }
            )
        )

    return pd.concat(
        actual_parts,
        ignore_index=True
    )


# ============================================================
# 6. Construct order-up-to targets
# ============================================================

def construct_targets(protection_period):

    relevant_daily_forecasts = forecasts[
        forecasts["horizon"] <= protection_period
    ].copy()

    assert (
        relevant_daily_forecasts
        .groupby(
            [
                "model",
                "id",
                "forecast_origin"
            ]
        )
        .size()
        .eq(protection_period)
        .all()
    )

    cumulative_forecasts = (
        relevant_daily_forecasts
        .groupby(
            [
                "model",
                "id",
                "demand_class",
                "forecast_origin"
            ],
            as_index=False
        )
        .agg(
            central_forecast=(
                "forecast",
                "sum"
            )
        )
    )

    scales = calculate_protection_scales(
        protection_period=protection_period,
        forecast_origins=(
            ERROR_ORIGINS
            + WEEKLY_ORIGINS
        )
    )

    errors = (
        cumulative_forecasts[
            cumulative_forecasts[
                "forecast_origin"
            ].isin(ERROR_ORIGINS)
        ]
        .merge(
            construct_actual_totals(
                protection_period
            ),
            on=[
                "id",
                "forecast_origin"
            ]
        )
        .merge(
            scales,
            on=[
                "id",
                "forecast_origin"
            ]
        )
    )

    errors["protection_error"] = (
        errors["actual_total"]
        - errors["central_forecast"]
    )

    errors["standardised_error"] = (
        errors["protection_error"]
        / errors["protection_scale"]
    )

    errors["protection_period"] = protection_period

    decision_forecasts = cumulative_forecasts[
        cumulative_forecasts[
            "forecast_origin"
        ].isin(WEEKLY_ORIGINS)
    ]

    target_parts = []

    for forecast_origin in WEEKLY_ORIGINS:

        # Only errors whose complete protection period was
        # observed by the current origin may enter the pool.
        available_errors = errors[
            (
                errors["forecast_origin"]
                + protection_period
            )
            <= forecast_origin
        ]

        quantile_rows = []

        for (
            model_name,
            demand_class
        ), group in available_errors.groupby(
            [
                "model",
                "demand_class"
            ]
        ):

            standardised_errors = (
                group["standardised_error"]
                .to_numpy()
            )

            for quantile_level in QUANTILE_LEVELS:

                quantile_rows.append(
                    {
                        "model": model_name,
                        "demand_class": demand_class,
                        "quantile_level": quantile_level,
                        "error_quantile": np.quantile(
                            standardised_errors,
                            quantile_level
                        ),
                        "pool_size": len(
                            standardised_errors
                        )
                    }
                )

        quantile_table = pd.DataFrame(
            quantile_rows
        )

        current_scales = scales[
            scales["forecast_origin"]
            == forecast_origin
        ][
            [
                "id",
                "protection_scale"
            ]
        ]

        current_targets = (
            decision_forecasts[
                decision_forecasts[
                    "forecast_origin"
                ]
                == forecast_origin
            ]
            .merge(
                current_scales,
                on="id"
            )
            .merge(
                quantile_table,
                on=[
                    "model",
                    "demand_class"
                ]
            )
        )

        current_targets["safety_stock"] = np.maximum(
            0.0,
            (
                current_targets["protection_scale"]
                * current_targets["error_quantile"]
            )
        )

        current_targets["order_up_to_target"] = np.maximum(
            0.0,
            (
                current_targets["central_forecast"]
                + current_targets["safety_stock"]
            )
        )

        current_targets[
            "protection_period"
        ] = protection_period

        target_parts.append(current_targets)

    targets = pd.concat(
        target_parts,
        ignore_index=True
    )

    assert targets[
        "order_up_to_target"
    ].notna().all()

    return targets, errors


all_target_parts = []
all_error_parts = []

# The six policies produce only three distinct
# protection periods: 14, 21 and 28 days.
for protection_period in [14, 21, 28]:

    protection_targets, protection_errors = (
        construct_targets(protection_period)
    )

    all_target_parts.append(protection_targets)
    all_error_parts.append(protection_errors)

    print(
        "Completed targets for "
        f"P={protection_period}."
    )


all_targets = pd.concat(
    all_target_parts,
    ignore_index=True
)

all_errors = pd.concat(
    all_error_parts,
    ignore_index=True
)

assert len(all_targets) == (
    3 * 5 * 3 * 500 * 16
)

all_targets.to_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_targets.csv",
    index=False
)

all_errors.to_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_errors.csv",
    index=False
)


# ============================================================
# 7. Prepare the target lookup
# ============================================================

target_lookup = {}

for key, group in all_targets.groupby(
    [
        "protection_period",
        "model",
        "quantile_level",
        "forecast_origin"
    ]
):

    target_vector = (
        group
        .set_index("id")[
            "order_up_to_target"
        ]
        .reindex(series_ids)
    )

    assert target_vector.notna().all()

    target_lookup[key] = (
        target_vector.to_numpy(dtype=float)
    )


# ============================================================
# 8. Inventory simulation
# ============================================================

def initial_pipeline_offsets(
    review_period,
    lead_time
):

    first_offset = (
        lead_time % review_period
        or review_period
    )

    return list(
        range(
            first_offset,
            lead_time + 1,
            review_period
        )
    )


def simulate_inventory_policy(
    review_period,
    lead_time,
    model_name,
    quantile_level
):

    protection_period = (
        review_period + lead_time
    )

    review_origins = list(
        range(
            1829,
            1935,
            review_period
        )
    )

    assert set(review_origins).issubset(
        WEEKLY_ORIGINS
    )

    # d_1829 is used for initialisation.
    review_days = set(review_origins[1:])

    recent_mean_demand = demand_matrix[
        :,
        1464:1829
    ].mean(axis=1)

    initial_pipeline = np.rint(
        recent_mean_demand * lead_time
    )

    initial_target = target_lookup[
        (
            protection_period,
            model_name,
            quantile_level,
            1829
        )
    ]

    inventory = np.maximum(
        0.0,
        initial_target - initial_pipeline
    )

    pipeline = (
        initial_pipeline
        .astype(float)
        .copy()
    )

    scheduled_arrivals = {}

    pipeline_offsets = initial_pipeline_offsets(
        review_period,
        lead_time
    )

    for offset in pipeline_offsets:

        scheduled_arrivals[
            1829 + offset
        ] = (
            initial_pipeline
            / len(pipeline_offsets)
        )

    holding_cost = np.zeros(number_of_series)
    ordering_cost = np.zeros(number_of_series)
    lost_sales_cost = np.zeros(number_of_series)

    demand_total = np.zeros(number_of_series)
    sales_total = np.zeros(number_of_series)
    lost_total = np.zeros(number_of_series)

    inventory_total = np.zeros(number_of_series)
    order_count = np.zeros(
        number_of_series,
        dtype=int
    )

    any_stockout = np.zeros(
        number_of_series,
        dtype=bool
    )

    for day in SIMULATION_DAYS:

        arrivals = scheduled_arrivals.pop(
            day,
            np.zeros(number_of_series)
        )

        available_inventory = (
            inventory + arrivals
        )

        daily_demand = demand_matrix[
            :,
            day - 1
        ]

        daily_sales = np.minimum(
            daily_demand,
            available_inventory
        )

        daily_lost_sales = np.maximum(
            0.0,
            daily_demand - available_inventory
        )

        ending_inventory = np.maximum(
            0.0,
            available_inventory - daily_demand
        )

        pipeline_before_order = np.maximum(
            0.0,
            pipeline - arrivals
        )

        order_quantity = np.zeros(
            number_of_series
        )

        if day in review_days:

            order_up_to_target = target_lookup[
                (
                    protection_period,
                    model_name,
                    quantile_level,
                    day
                )
            ]

            order_quantity = np.maximum(
                0.0,
                (
                    order_up_to_target
                    - ending_inventory
                    - pipeline_before_order
                )
            )

            arrival_day = day + lead_time

            scheduled_arrivals[arrival_day] = (
                scheduled_arrivals.get(
                    arrival_day,
                    np.zeros(number_of_series)
                )
                + order_quantity
            )

        closing_pipeline = (
            pipeline_before_order
            + order_quantity
        )

        assert np.allclose(
            daily_demand,
            daily_sales + daily_lost_sales
        )

        if day >= FINAL_TEST_START:

            daily_price = price_matrix[
                :,
                day - SIMULATION_DAYS[0]
            ]

            order_indicator = (
                order_quantity > 1e-10
            )

            holding_cost += (
                ANNUAL_HOLDING_RATE
                / 365
                * ending_inventory
                * daily_price
            )

            ordering_cost += (
                FIXED_ORDERING_COST
                * order_indicator
            )

            lost_sales_cost += (
                LOST_SALES_RATE
                * daily_lost_sales
                * daily_price
            )

            demand_total += daily_demand
            sales_total += daily_sales
            lost_total += daily_lost_sales
            inventory_total += ending_inventory

            order_count += (
                order_indicator.astype(int)
            )

            any_stockout |= (
                daily_lost_sales > 1e-10
            )

        inventory = ending_inventory
        pipeline = closing_pipeline

    return pd.DataFrame(
        {
            "review_period": review_period,
            "lead_time": lead_time,
            "protection_period": protection_period,
            "model": model_name,
            "quantile_level": quantile_level,
            "id": series_ids,
            "demand_class": (
                sample["demand_class"].to_numpy()
            ),
            "holding_cost": holding_cost,
            "ordering_cost": ordering_cost,
            "lost_sales_cost": lost_sales_cost,
            "total_cost": (
                holding_cost
                + ordering_cost
                + lost_sales_cost
            ),
            "demand_units": demand_total,
            "sales_units": sales_total,
            "lost_units": lost_total,
            "cycle_service_indicator": (
                ~any_stockout
            ).astype(int),
            "average_inventory": (
                inventory_total / 28
            ),
            "order_count": order_count,
            "ending_inventory": inventory,
            "ending_pipeline": pipeline
        }
    )


simulation_parts = []

for review_period, lead_time in POLICIES:

    for model_name in MODELS:

        for quantile_level in QUANTILE_LEVELS:

            policy_results = simulate_inventory_policy(
                review_period=review_period,
                lead_time=lead_time,
                model_name=model_name,
                quantile_level=quantile_level
            )

            simulation_parts.append(
                policy_results
            )

            print(
                f"Completed R={review_period}, "
                f"L={lead_time}, "
                f"{model_name}, "
                f"q={quantile_level:.2f}"
            )


series_results = pd.concat(
    simulation_parts,
    ignore_index=True
)

assert len(series_results) == (
    6 * 5 * 3 * 500
)

series_results.to_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_series_results.csv",
    index=False
)


# ============================================================
# 9. Summarise the results
# ============================================================

BASE_GROUPS = [
    "review_period",
    "lead_time",
    "protection_period",
    "model",
    "quantile_level"
]


def summarise_inventory_results(
    data,
    by_demand_class
):

    grouping_columns = (
        BASE_GROUPS
        + (
            ["demand_class"]
            if by_demand_class
            else []
        )
    )

    summary_rows = []

    for key, group in data.groupby(
        grouping_columns
    ):

        row = dict(
            zip(grouping_columns, key)
        )

        if not by_demand_class:
            row["demand_class"] = "Overall"

        total_demand = (
            group["demand_units"].sum()
        )

        total_sales = (
            group["sales_units"].sum()
        )

        row.update(
            {
                "number_of_series": (
                    group["id"].nunique()
                ),
                "holding_cost": (
                    group["holding_cost"].sum()
                ),
                "ordering_cost": (
                    group["ordering_cost"].sum()
                ),
                "lost_sales_cost": (
                    group["lost_sales_cost"].sum()
                ),
                "total_cost": (
                    group["total_cost"].sum()
                ),
                "fill_rate": (
                    total_sales / total_demand
                ),
                "cycle_service_level": (
                    group[
                        "cycle_service_indicator"
                    ].mean()
                ),
                "average_inventory": (
                    group[
                        "average_inventory"
                    ].mean()
                ),
                "lost_units": (
                    group["lost_units"].sum()
                ),
                "order_count": (
                    group["order_count"].sum()
                )
            }
        )

        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


inventory_summary = pd.concat(
    [
        summarise_inventory_results(
            series_results,
            by_demand_class=False
        ),
        summarise_inventory_results(
            series_results,
            by_demand_class=True
        )
    ],
    ignore_index=True
)

inventory_summary[
    "meets_minimum_fill_rate"
] = (
    inventory_summary["fill_rate"]
    >= MINIMUM_FILL_RATE
)

inventory_summary[
    "eligible_cost_rank"
] = np.nan

eligible_results = inventory_summary[
    inventory_summary[
        "meets_minimum_fill_rate"
    ]
]

inventory_summary.loc[
    eligible_results.index,
    "eligible_cost_rank"
] = (
    eligible_results
    .groupby(
        [
            "review_period",
            "lead_time",
            "demand_class",
            "quantile_level"
        ]
    )["total_cost"]
    .rank(
        method="min",
        ascending=True
    )
)


# ============================================================
# 10. Compare every policy with the original R=7, L=21 policy
# ============================================================

baseline_results = inventory_summary[
    (
        inventory_summary["review_period"] == 7
    )
    & (
        inventory_summary["lead_time"] == 21
    )
][
    [
        "model",
        "quantile_level",
        "demand_class",
        "total_cost",
        "fill_rate"
    ]
].rename(
    columns={
        "total_cost": "baseline_total_cost",
        "fill_rate": "baseline_fill_rate"
    }
)

policy_comparison = inventory_summary.merge(
    baseline_results,
    on=[
        "model",
        "quantile_level",
        "demand_class"
    ],
    validate="many_to_one"
)

policy_comparison[
    "cost_difference_vs_baseline"
] = (
    policy_comparison["total_cost"]
    - policy_comparison["baseline_total_cost"]
)

policy_comparison[
    "fill_rate_difference_pp_vs_baseline"
] = 100 * (
    policy_comparison["fill_rate"]
    - policy_comparison["baseline_fill_rate"]
)


# ============================================================
# 11. Implementation checks
# ============================================================

implementation_checks = pd.DataFrame(
    {
        "check": [
            "Six R-L combinations",
            "R plus L never exceeds 28",
            "Expected protection-error rows",
            "Positive finite protection scales",
            "Finite standardised errors",
            "Expected target rows",
            "Non-negative safety stock",
            "Target not below central forecast",
            "Expected series rows",
            "Exactly five models",
            "Exactly three quantile levels",
            "No negative costs",
            "Fill rates between zero and one"
        ],
        "passed": [
            len(POLICIES) == 6,

            all(
                review_period + lead_time <= 28
                for review_period, lead_time
                in POLICIES
            ),

            len(all_errors)
            == 3 * 5 * 500 * 9,

            (
                np.isfinite(
                    all_errors["protection_scale"]
                ).all()
                and (
                    all_errors["protection_scale"] > 0
                ).all()
            ),

            np.isfinite(
                all_errors["standardised_error"]
            ).all(),

            len(all_targets)
            == 3 * 5 * 3 * 500 * 16,

            (
                all_targets["safety_stock"] >= 0
            ).all(),

            (
                all_targets["order_up_to_target"]
                >= all_targets["central_forecast"]
            ).all(),

            len(series_results) == 45000,

            (
                series_results["model"].nunique()
                == 5
            ),

            (
                series_results[
                    "quantile_level"
                ].nunique()
                == 3
            ),

            (
                inventory_summary["total_cost"] >= 0
            ).all(),

            inventory_summary[
                "fill_rate"
            ].between(0, 1).all()
        ]
    }
)

if not implementation_checks["passed"].all():

    failed_checks = implementation_checks.loc[
        ~implementation_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Implementation checks failed: {failed_checks}"
    )


# ============================================================
# 12. Save the final files
# ============================================================

inventory_summary.to_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_summary.csv",
    index=False
)

policy_comparison.to_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_comparison_vs_baseline.csv",
    index=False
)

implementation_checks.to_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_checks.csv",
    index=False
)


print("\nWeekly R-L sensitivity analysis completed successfully.\n")

print("Implementation checks:")
print(implementation_checks)

best_overall_results = (
    inventory_summary[
        (
            inventory_summary["demand_class"]
            == "Overall"
        )
        & inventory_summary[
            "meets_minimum_fill_rate"
        ]
    ]
    .sort_values("total_cost")
    .groupby(
        [
            "review_period",
            "lead_time",
            "quantile_level"
        ],
        as_index=False
    )
    .first()
)

print(
    "\nLowest-cost eligible model "
    "for each policy and quantile:"
)

print(
    best_overall_results[
        [
            "review_period",
            "lead_time",
            "protection_period",
            "model",
            "quantile_level",
            "total_cost",
            "fill_rate",
            "average_inventory",
            "order_count",
            "eligible_cost_rank"
        ]
    ]
)

print("\nSaved files:")

for file_name in [
    "inventory_policy_timing_targets.csv",
    "inventory_policy_timing_errors.csv",
    "inventory_policy_timing_series_results.csv",
    "inventory_policy_timing_summary.csv",
    "inventory_policy_timing_comparison_vs_baseline.csv",
    "inventory_policy_timing_checks.csv"
]:
    print(OUTPUT_DIR / file_name)


Demand classes recovered from: inventory_weekly_order_up_to_targets.csv

Demand classes:
demand_class
Erratic         125
Intermittent    125
Lumpy           125
Smooth          125
Name: count, dtype: int64
Series whose price-based active period begins after d_1: 246

Selected series: 500
Observed demand ends at: d_1941
Forecast rows loaded: 1540000
Completed targets for P=14.
Completed targets for P=21.
Completed targets for P=28.
Completed R=7, L=7, seasonal_naive, q=0.90
Completed R=7, L=7, seasonal_naive, q=0.95
Completed R=7, L=7, seasonal_naive, q=0.99
Completed R=7, L=7, ets, q=0.90
Completed R=7, L=7, ets, q=0.95
Completed R=7, L=7, ets, q=0.99
Completed R=7, L=7, arima_sarima, q=0.90
Completed R=7, L=7, arima_sarima, q=0.95
Completed R=7, L=7, arima_sarima, q=0.99
Completed R=7, L=7, xgboost, q=0.90
Completed R=7, L=7, xgboost, q=0.95
Completed R=7, L=7, xgboost, q=0.99
Completed R=7, L=7, tirex2, q=0.90
Completed R=7, L=7, tirex2, q=0.95
Completed R=7, L=7, tirex2, q=0.99
Co

## 2. Verify reconstruction of the original baseline

This diagnostic compares the reconstructed `(R,L)=(7,21)` results with the original inventory summary.


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))
OUTPUT_DIR = DATA_DIR / "chapter5_results"

old_summary = pd.read_csv(
    OUTPUT_DIR / "inventory_summary.csv"
)

new_summary = pd.read_csv(
    OUTPUT_DIR / "inventory_policy_timing_summary.csv"
)


def normalise_model_names(series):
    return (
        series.astype(str)
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .replace(
            {
                "tirex_2": "tirex2",
                "seasonal_naïve": "seasonal_naive"
            }
        )
    )


old_summary["model"] = normalise_model_names(
    old_summary["model"]
)

new_summary["model"] = normalise_model_names(
    new_summary["model"]
)


# Select the original baseline scenario
if "scenario" in old_summary.columns:
    old_baseline = old_summary[
        old_summary["scenario"]
        .astype(str)
        .str.lower()
        .eq("baseline")
    ].copy()
else:
    old_baseline = old_summary.copy()


# Select the overall results
if "demand_class" in old_baseline.columns:

    old_baseline["demand_class"] = (
        old_baseline["demand_class"]
        .astype(str)
        .str.strip()
    )

    old_baseline = old_baseline[
        old_baseline["demand_class"]
        .str.lower()
        .eq("overall")
    ].copy()


new_baseline = new_summary[
    (new_summary["review_period"] == 7)
    & (new_summary["lead_time"] == 21)
    & (
        new_summary["demand_class"]
        .astype(str)
        .str.lower()
        .eq("overall")
    )
].copy()


metrics = [
    "holding_cost",
    "ordering_cost",
    "lost_sales_cost",
    "total_cost",
    "fill_rate",
    "cycle_service_level",
    "average_inventory",
    "order_count"
]

comparison = old_baseline[
    ["model", "quantile_level"] + metrics
].merge(
    new_baseline[
        ["model", "quantile_level"] + metrics
    ],
    on=[
        "model",
        "quantile_level"
    ],
    suffixes=(
        "_old",
        "_new"
    ),
    validate="one_to_one"
)


for metric in metrics:
    comparison[
        f"{metric}_difference"
    ] = (
        comparison[f"{metric}_new"]
        - comparison[f"{metric}_old"]
    )


print(
    comparison[
        [
            "model",
            "quantile_level",
            "total_cost_old",
            "total_cost_new",
            "total_cost_difference",
            "fill_rate_old",
            "fill_rate_new",
            "fill_rate_difference",
            "order_count_old",
            "order_count_new",
            "order_count_difference"
        ]
    ].sort_values(
        [
            "quantile_level",
            "model"
        ]
    )
)


# Also compare the reconstructed 28-day targets
old_targets = pd.read_csv(
    OUTPUT_DIR
    / "inventory_weekly_order_up_to_targets.csv"
)

new_targets = pd.read_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_targets.csv"
)

old_targets["model"] = normalise_model_names(
    old_targets["model"]
)

new_targets["model"] = normalise_model_names(
    new_targets["model"]
)

new_targets = new_targets[
    new_targets["protection_period"] == 28
].copy()

target_comparison = old_targets[
    [
        "id",
        "model",
        "quantile_level",
        "forecast_origin",
        "order_up_to_target"
    ]
].merge(
    new_targets[
        [
            "id",
            "model",
            "quantile_level",
            "forecast_origin",
            "order_up_to_target"
        ]
    ],
    on=[
        "id",
        "model",
        "quantile_level",
        "forecast_origin"
    ],
    suffixes=(
        "_old",
        "_new"
    ),
    validate="one_to_one"
)

target_comparison[
    "absolute_target_difference"
] = np.abs(
    target_comparison[
        "order_up_to_target_new"
    ]
    - target_comparison[
        "order_up_to_target_old"
    ]
)

print(
    "\nMaximum absolute difference between "
    "old and reconstructed 28-day targets:"
)

print(
    target_comparison[
        "absolute_target_difference"
    ].max()
)


             model  quantile_level  total_cost_old  total_cost_new  \
2     arima_sarima            0.90     1146.913816     1147.567860   
1              ets            0.90     1047.321699     1047.289527   
0   seasonal_naive            0.90      805.816118      805.836943   
4           tirex2            0.90     1220.570293     1220.845173   
3          xgboost            0.90     1182.252159     1182.435611   
8     arima_sarima            0.95      967.705944      967.541085   
6              ets            0.95      891.083340      891.771597   
5   seasonal_naive            0.95      735.582976      735.735793   
9           tirex2            0.95     1026.255853     1026.679110   
7          xgboost            0.95      917.188693      917.168939   
13    arima_sarima            0.99      791.588274      791.591624   
12             ets            0.99      743.732234      743.523435   
10  seasonal_naive            0.99      641.326425      638.963827   
14          tirex2  

## 3. Compare original and reconstructed 28-day targets

This diagnostic isolates any differences in protection scales, safety stocks and order-up-to targets.


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))
OUTPUT_DIR = DATA_DIR / "chapter5_results"

old_targets = pd.read_csv(
    OUTPUT_DIR
    / "inventory_weekly_order_up_to_targets.csv"
)

new_targets = pd.read_csv(
    OUTPUT_DIR
    / "inventory_policy_timing_targets.csv"
)

new_targets = new_targets[
    new_targets["protection_period"] == 28
].copy()


def normalise_model_names(series):
    return (
        series.astype(str)
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .replace(
            {
                "tirex_2": "tirex2",
                "seasonal_naïve": "seasonal_naive"
            }
        )
    )


old_targets["model"] = normalise_model_names(
    old_targets["model"]
)

new_targets["model"] = normalise_model_names(
    new_targets["model"]
)


key_columns = [
    "id",
    "model",
    "quantile_level",
    "forecast_origin"
]

possible_components = [
    "central_forecast",
    "protection_scale",
    "error_quantile",
    "safety_stock",
    "order_up_to_target",
    "pool_size"
]

common_components = [
    column
    for column in possible_components
    if (
        column in old_targets.columns
        and column in new_targets.columns
    )
]

print("Old target columns:")
print(old_targets.columns.tolist())

print("\nNew target columns:")
print(new_targets.columns.tolist())

print("\nComparable components:")
print(common_components)


target_comparison = old_targets[
    key_columns + common_components
].merge(
    new_targets[
        key_columns + common_components
    ],
    on=key_columns,
    suffixes=(
        "_old",
        "_new"
    ),
    validate="one_to_one"
)


diagnostic_rows = []

for component in common_components:

    old_values = target_comparison[
        f"{component}_old"
    ].astype(float)

    new_values = target_comparison[
        f"{component}_new"
    ].astype(float)

    absolute_difference = np.abs(
        new_values - old_values
    )

    diagnostic_rows.append(
        {
            "component": component,
            "maximum_absolute_difference": (
                absolute_difference.max()
            ),
            "mean_absolute_difference": (
                absolute_difference.mean()
            ),
            "different_rows": (
                absolute_difference > 1e-10
            ).sum(),
            "total_rows": len(
                absolute_difference
            )
        }
    )


diagnostic_summary = pd.DataFrame(
    diagnostic_rows
)

print("\nComponent comparison:")
print(diagnostic_summary)


target_comparison[
    "absolute_target_difference"
] = np.abs(
    target_comparison[
        "order_up_to_target_new"
    ]
    - target_comparison[
        "order_up_to_target_old"
    ]
)


columns_to_show = key_columns.copy()

for component in common_components:
    columns_to_show.extend(
        [
            f"{component}_old",
            f"{component}_new"
        ]
    )

columns_to_show.append(
    "absolute_target_difference"
)


largest_differences = (
    target_comparison
    .sort_values(
        "absolute_target_difference",
        ascending=False
    )
    .head(10)
)


print("\nTen largest target differences:")
print(
    largest_differences[
        columns_to_show
    ].to_string(index=False)
)


Old target columns:
['id', 'demand_class', 'model', 'forecast_origin', 'forecast_protection_demand', 'forecast_days', 'protection_scale', 'quantile_level', 'standardised_error_quantile', 'pool_size', 'safety_stock', 'order_up_to_target']

New target columns:
['model', 'id', 'demand_class', 'forecast_origin', 'central_forecast', 'protection_scale', 'quantile_level', 'error_quantile', 'pool_size', 'safety_stock', 'order_up_to_target', 'protection_period']

Comparable components:
['protection_scale', 'safety_stock', 'order_up_to_target', 'pool_size']

Component comparison:
            component  maximum_absolute_difference  mean_absolute_difference  \
0    protection_scale                          0.0                       0.0   
1        safety_stock                          0.0                       0.0   
2  order_up_to_target                          0.0                       0.0   
3           pool_size                          0.0                       0.0   

   different_rows  tot

## 4. Confirm the source of protection-scale differences

This cell checks complete-history and active-history scale calculations. It depends on objects created in the first cell.


In [16]:
# ============================================================
# Confirm the source of the protection-scale difference
# ============================================================

old_scale_table = (
    old_targets[
        [
            "id",
            "forecast_origin",
            "protection_scale"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "protection_scale":
            "old_protection_scale"
        }
    )
)

protection_period = 28

cumulative_demand = np.pad(
    np.cumsum(
        demand_matrix,
        axis=1
    ),
    (
        (0, 0),
        (1, 0)
    )
)

rolling_28_day_demand = (
    cumulative_demand[:, protection_period:]
    - cumulative_demand[:, :-protection_period]
)

comparison_parts = []

for forecast_origin in WEEKLY_ORIGINS:

    history_end = (
        forecast_origin
        - protection_period
        + 1
    )

    full_history_scales = np.empty(
        number_of_series
    )

    active_history_scales = np.empty(
        number_of_series
    )

    for series_index in range(
        number_of_series
    ):

        # Method 1: include the complete calendar history
        full_history = rolling_28_day_demand[
            series_index,
            :history_end
        ]

        full_standard_deviation = np.std(
            full_history,
            ddof=1
        )

        full_history_scales[
            series_index
        ] = max(
            1.0,
            full_standard_deviation
        )

        # Method 2: begin at the first active day
        active_history = rolling_28_day_demand[
            series_index,
            first_active_day[series_index] - 1:
            history_end
        ]

        active_standard_deviation = np.std(
            active_history,
            ddof=1
        )

        active_history_scales[
            series_index
        ] = max(
            1.0,
            active_standard_deviation
        )

    comparison_parts.append(
        pd.DataFrame(
            {
                "id": series_ids,
                "forecast_origin": forecast_origin,
                "full_history_scale": (
                    full_history_scales
                ),
                "active_history_scale": (
                    active_history_scales
                )
            }
        )
    )


scale_method_comparison = pd.concat(
    comparison_parts,
    ignore_index=True
)

scale_method_comparison = (
    old_scale_table
    .merge(
        scale_method_comparison,
        on=[
            "id",
            "forecast_origin"
        ],
        validate="one_to_one"
    )
)

scale_method_comparison[
    "old_vs_full_difference"
] = np.abs(
    scale_method_comparison[
        "old_protection_scale"
    ]
    - scale_method_comparison[
        "full_history_scale"
    ]
)

scale_method_comparison[
    "old_vs_active_difference"
] = np.abs(
    scale_method_comparison[
        "old_protection_scale"
    ]
    - scale_method_comparison[
        "active_history_scale"
    ]
)


print(
    "Series with first active day after d_1:",
    (first_active_day > 1).sum()
)

print(
    "\nMaximum difference: old versus full history:",
    scale_method_comparison[
        "old_vs_full_difference"
    ].max()
)

print(
    "Maximum difference: old versus active history:",
    scale_method_comparison[
        "old_vs_active_difference"
    ].max()
)

print(
    "\nMean difference: old versus full history:",
    scale_method_comparison[
        "old_vs_full_difference"
    ].mean()
)

print(
    "Mean difference: old versus active history:",
    scale_method_comparison[
        "old_vs_active_difference"
    ].mean()
)


Series with first active day after d_1: 246

Maximum difference: old versus full history: 98.89535599414145
Maximum difference: old versus active history: 5.684341886080802e-14

Mean difference: old versus full history: 4.331849656355181
Mean difference: old versus active history: 8.962275366286576e-16


## 5. Paired bootstrap for timing alternatives

This cell compares each timing policy with the `(7,21)` baseline using 2,000 paired series resamples.


In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# 1. Settings
# ============================================================

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))
OUTPUT_DIR = DATA_DIR / "chapter5_results"

RESULT_FILE = (
    OUTPUT_DIR
    / "inventory_policy_timing_series_results.csv"
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "inventory_policy_timing_paired_bootstrap.csv"
)

CHECK_FILE = (
    OUTPUT_DIR
    / "inventory_policy_timing_bootstrap_checks.csv"
)

BOOTSTRAP_RESAMPLES = 2000
RANDOM_SEED = 2026

BASELINE_POLICY = (7, 21)

POLICIES = [
    (7, 7),
    (7, 14),
    (7, 21),
    (14, 7),
    (14, 14),
    (21, 7)
]

ALTERNATIVE_POLICIES = [
    policy
    for policy in POLICIES
    if policy != BASELINE_POLICY
]

MODELS = [
    "seasonal_naive",
    "ets",
    "arima_sarima",
    "xgboost",
    "tirex2"
]

QUANTILE_LEVELS = [
    0.90,
    0.95,
    0.99
]


def normalise_model_names(series):

    return (
        series.astype(str)
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .replace(
            {
                "tirex_2": "tirex2",
                "seasonal_naïve": "seasonal_naive"
            }
        )
    )


# ============================================================
# 2. Load the per-series inventory results
# ============================================================

results = pd.read_csv(RESULT_FILE)

results["id"] = results["id"].astype(str)

results["model"] = normalise_model_names(
    results["model"]
)

results["review_period"] = (
    results["review_period"].astype(int)
)

results["lead_time"] = (
    results["lead_time"].astype(int)
)

results["quantile_level"] = (
    results["quantile_level"].astype(float)
)

results["demand_class"] = (
    results["demand_class"]
    .astype(str)
    .str.strip()
    .str.capitalize()
)


required_columns = {
    "review_period",
    "lead_time",
    "model",
    "quantile_level",
    "id",
    "demand_class",
    "holding_cost",
    "ordering_cost",
    "lost_sales_cost",
    "total_cost",
    "demand_units",
    "sales_units",
    "average_inventory",
    "order_count"
}

missing_columns = (
    required_columns
    - set(results.columns)
)

if missing_columns:
    raise KeyError(
        f"Missing columns: {sorted(missing_columns)}"
    )


key_columns = [
    "review_period",
    "lead_time",
    "model",
    "quantile_level",
    "id"
]

if results.duplicated(key_columns).any():
    raise ValueError(
        "Duplicate policy/model/quantile/series "
        "rows were found."
    )


assert len(results) == (
    6 * 5 * 3 * 500
)

assert set(
    zip(
        results["review_period"],
        results["lead_time"]
    )
) == set(POLICIES)

assert set(
    results["model"]
) == set(MODELS)


# ============================================================
# 3. Define the overall sample and demand groups
# ============================================================

group_ids = {
    "Overall": sorted(
        results["id"].unique()
    )
}

for demand_class in [
    "Smooth",
    "Erratic",
    "Intermittent",
    "Lumpy"
]:

    group_ids[demand_class] = sorted(
        results.loc[
            results["demand_class"]
            == demand_class,
            "id"
        ].unique()
    )


expected_group_sizes = {
    "Overall": 500,
    "Smooth": 125,
    "Erratic": 125,
    "Intermittent": 125,
    "Lumpy": 125
}

actual_group_sizes = {
    group_name: len(ids)
    for group_name, ids
    in group_ids.items()
}

assert (
    actual_group_sizes
    == expected_group_sizes
)

print("Series per demand group:")
print(actual_group_sizes)


# ============================================================
# 4. Generate fixed paired bootstrap samples
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_indices = {
    group_name: rng.integers(
        low=0,
        high=len(ids),
        size=(
            BOOTSTRAP_RESAMPLES,
            len(ids)
        )
    )
    for group_name, ids
    in group_ids.items()
}


# ============================================================
# 5. Paired bootstrap comparisons
# ============================================================

comparison_rows = []

for demand_group, ids in group_ids.items():

    sampled_indices = (
        bootstrap_indices[demand_group]
    )

    for model_name in MODELS:

        for quantile_level in QUANTILE_LEVELS:

            baseline = results[
                (
                    results["review_period"]
                    == BASELINE_POLICY[0]
                )
                & (
                    results["lead_time"]
                    == BASELINE_POLICY[1]
                )
                & (
                    results["model"]
                    == model_name
                )
                & np.isclose(
                    results["quantile_level"],
                    quantile_level
                )
                & results["id"].isin(ids)
            ].set_index("id").reindex(ids)

            assert len(baseline) == len(ids)

            assert baseline[
                "total_cost"
            ].notna().all()


            for (
                alternative_review_period,
                alternative_lead_time
            ) in ALTERNATIVE_POLICIES:

                alternative = results[
                    (
                        results["review_period"]
                        == alternative_review_period
                    )
                    & (
                        results["lead_time"]
                        == alternative_lead_time
                    )
                    & (
                        results["model"]
                        == model_name
                    )
                    & np.isclose(
                        results["quantile_level"],
                        quantile_level
                    )
                    & results["id"].isin(ids)
                ].set_index("id").reindex(ids)

                assert len(alternative) == len(ids)

                assert alternative[
                    "total_cost"
                ].notna().all()


                baseline_demand = (
                    baseline["demand_units"]
                    .to_numpy(dtype=float)
                )

                alternative_demand = (
                    alternative["demand_units"]
                    .to_numpy(dtype=float)
                )

                # The same realised demand must be used
                # under both policies.
                assert np.allclose(
                    baseline_demand,
                    alternative_demand
                )


                cost_difference = (
                    alternative["total_cost"]
                    .to_numpy(dtype=float)
                    - baseline["total_cost"]
                    .to_numpy(dtype=float)
                )

                sales_difference = (
                    alternative["sales_units"]
                    .to_numpy(dtype=float)
                    - baseline["sales_units"]
                    .to_numpy(dtype=float)
                )


                # --------------------------------------------
                # Bootstrap total-cost difference
                # --------------------------------------------

                bootstrap_cost_difference = (
                    cost_difference[
                        sampled_indices
                    ].sum(axis=1)
                )


                # --------------------------------------------
                # Bootstrap fill-rate difference
                # --------------------------------------------

                bootstrap_demand = (
                    baseline_demand[
                        sampled_indices
                    ].sum(axis=1)
                )

                if np.any(
                    bootstrap_demand <= 0
                ):
                    raise ValueError(
                        "A bootstrap resample has "
                        "no observed demand."
                    )

                bootstrap_fill_difference_pp = (
                    100
                    * (
                        sales_difference[
                            sampled_indices
                        ].sum(axis=1)
                        / bootstrap_demand
                    )
                )


                # --------------------------------------------
                # Point estimates
                # --------------------------------------------

                point_cost_difference = (
                    cost_difference.sum()
                )

                baseline_total_cost = (
                    baseline["total_cost"].sum()
                )

                alternative_total_cost = (
                    alternative["total_cost"].sum()
                )

                total_observed_demand = (
                    baseline_demand.sum()
                )

                baseline_fill_rate = (
                    baseline["sales_units"].sum()
                    / total_observed_demand
                )

                alternative_fill_rate = (
                    alternative["sales_units"].sum()
                    / total_observed_demand
                )

                point_fill_difference_pp = (
                    100
                    * (
                        alternative_fill_rate
                        - baseline_fill_rate
                    )
                )


                # --------------------------------------------
                # 95% percentile confidence intervals
                # --------------------------------------------

                (
                    cost_ci_lower,
                    cost_ci_upper
                ) = np.quantile(
                    bootstrap_cost_difference,
                    [
                        0.025,
                        0.975
                    ]
                )

                (
                    fill_ci_lower,
                    fill_ci_upper
                ) = np.quantile(
                    bootstrap_fill_difference_pp,
                    [
                        0.025,
                        0.975
                    ]
                )


                alternative_label = (
                    f"R{alternative_review_period}"
                    f"_L{alternative_lead_time}"
                )

                baseline_label = "R7_L21"


                # Negative cost difference means that
                # the alternative policy is cheaper.
                if cost_ci_upper < 0:
                    lower_cost_policy = (
                        alternative_label
                    )

                elif cost_ci_lower > 0:
                    lower_cost_policy = (
                        baseline_label
                    )

                else:
                    lower_cost_policy = (
                        "no_clear_difference"
                    )


                # Positive fill-rate difference means that
                # the alternative has the higher fill rate.
                if fill_ci_lower > 0:
                    higher_fill_policy = (
                        alternative_label
                    )

                elif fill_ci_upper < 0:
                    higher_fill_policy = (
                        baseline_label
                    )

                else:
                    higher_fill_policy = (
                        "no_clear_difference"
                    )


                comparison_rows.append(
                    {
                        "demand_group":
                            demand_group,

                        "model":
                            model_name,

                        "quantile_level":
                            quantile_level,

                        "baseline_review_period":
                            BASELINE_POLICY[0],

                        "baseline_lead_time":
                            BASELINE_POLICY[1],

                        "alternative_review_period":
                            alternative_review_period,

                        "alternative_lead_time":
                            alternative_lead_time,

                        "alternative_protection_period":
                            (
                                alternative_review_period
                                + alternative_lead_time
                            ),

                        "series_count":
                            len(ids),

                        "bootstrap_resamples":
                            BOOTSTRAP_RESAMPLES,

                        "baseline_total_cost":
                            baseline_total_cost,

                        "alternative_total_cost":
                            alternative_total_cost,

                        "point_total_cost_difference_"
                        "alternative_minus_baseline":
                            point_cost_difference,

                        "percent_cost_change_"
                        "vs_baseline":
                            (
                                100
                                * point_cost_difference
                                / baseline_total_cost
                            ),

                        "bootstrap_mean_cost_difference":
                            bootstrap_cost_difference.mean(),

                        "bootstrap_se_cost_difference":
                            bootstrap_cost_difference.std(
                                ddof=1
                            ),

                        "cost_ci_lower":
                            cost_ci_lower,

                        "cost_ci_upper":
                            cost_ci_upper,

                        "cost_ci_excludes_zero":
                            bool(
                                cost_ci_lower > 0
                                or cost_ci_upper < 0
                            ),

                        "lower_cost_policy":
                            lower_cost_policy,

                        "baseline_fill_rate":
                            baseline_fill_rate,

                        "alternative_fill_rate":
                            alternative_fill_rate,

                        "point_fill_rate_difference_pp_"
                        "alternative_minus_baseline":
                            point_fill_difference_pp,

                        "bootstrap_mean_fill_"
                        "difference_pp":
                            (
                                bootstrap_fill_difference_pp
                                .mean()
                            ),

                        "bootstrap_se_fill_"
                        "difference_pp":
                            (
                                bootstrap_fill_difference_pp
                                .std(ddof=1)
                            ),

                        "fill_ci_lower_pp":
                            fill_ci_lower,

                        "fill_ci_upper_pp":
                            fill_ci_upper,

                        "fill_ci_excludes_zero":
                            bool(
                                fill_ci_lower > 0
                                or fill_ci_upper < 0
                            ),

                        "higher_fill_policy":
                            higher_fill_policy,

                        "holding_cost_difference":
                            (
                                alternative[
                                    "holding_cost"
                                ].sum()
                                - baseline[
                                    "holding_cost"
                                ].sum()
                            ),

                        "ordering_cost_difference":
                            (
                                alternative[
                                    "ordering_cost"
                                ].sum()
                                - baseline[
                                    "ordering_cost"
                                ].sum()
                            ),

                        "lost_sales_cost_difference":
                            (
                                alternative[
                                    "lost_sales_cost"
                                ].sum()
                                - baseline[
                                    "lost_sales_cost"
                                ].sum()
                            ),

                        "average_inventory_difference":
                            (
                                alternative[
                                    "average_inventory"
                                ].mean()
                                - baseline[
                                    "average_inventory"
                                ].mean()
                            ),

                        "order_count_difference":
                            (
                                alternative[
                                    "order_count"
                                ].sum()
                                - baseline[
                                    "order_count"
                                ].sum()
                            )
                    }
                )

    print(
        "Completed paired bootstrap for",
        demand_group
    )


bootstrap_results = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 6. Implementation checks
# ============================================================

bootstrap_checks = pd.DataFrame(
    {
        "check": [
            "Expected comparison rows",
            "Exactly 2000 bootstrap resamples",
            "Correct series counts",
            "Exactly five alternative policies",
            "Exactly five models",
            "Exactly three quantile levels",
            "Exactly five demand groups",
            "Finite bootstrap results",
            "Cost interval lower bound not above upper bound",
            "Fill interval lower bound not above upper bound"
        ],

        "passed": [
            len(bootstrap_results)
            == 5 * 5 * 3 * 5,

            bootstrap_results[
                "bootstrap_resamples"
            ].eq(
                BOOTSTRAP_RESAMPLES
            ).all(),

            bootstrap_results.apply(
                lambda row:
                    row["series_count"]
                    == expected_group_sizes[
                        row["demand_group"]
                    ],
                axis=1
            ).all(),

            bootstrap_results[
                [
                    "alternative_review_period",
                    "alternative_lead_time"
                ]
            ].drop_duplicates().shape[0]
            == 5,

            bootstrap_results[
                "model"
            ].nunique()
            == 5,

            bootstrap_results[
                "quantile_level"
            ].nunique()
            == 3,

            bootstrap_results[
                "demand_group"
            ].nunique()
            == 5,

            np.isfinite(
                bootstrap_results.select_dtypes(
                    include=[np.number]
                )
            ).all().all(),

            (
                bootstrap_results[
                    "cost_ci_lower"
                ]
                <= bootstrap_results[
                    "cost_ci_upper"
                ]
            ).all(),

            (
                bootstrap_results[
                    "fill_ci_lower_pp"
                ]
                <= bootstrap_results[
                    "fill_ci_upper_pp"
                ]
            ).all()
        ]
    }
)


if not bootstrap_checks["passed"].all():

    failed_checks = bootstrap_checks.loc[
        ~bootstrap_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Bootstrap checks failed: {failed_checks}"
    )


# ============================================================
# 7. Save and display the results
# ============================================================

bootstrap_results.to_csv(
    OUTPUT_FILE,
    index=False
)

bootstrap_checks.to_csv(
    CHECK_FILE,
    index=False
)


print(
    "\nPolicy-timing paired bootstrap "
    "completed successfully.\n"
)

print(
    "Comparison rows:",
    len(bootstrap_results)
)

print("\nBootstrap checks:")
print(bootstrap_checks)


overall_q99 = bootstrap_results[
    (
        bootstrap_results["demand_group"]
        == "Overall"
    )
    & np.isclose(
        bootstrap_results["quantile_level"],
        0.99
    )
].sort_values(
    [
        "model",
        "alternative_review_period",
        "alternative_lead_time"
    ]
)


print(
    "\nOverall comparisons with "
    "R=7, L=21 at q=0.99:"
)

print(
    overall_q99[
        [
            "model",
            "alternative_review_period",
            "alternative_lead_time",
            "point_total_cost_difference_"
            "alternative_minus_baseline",
            "cost_ci_lower",
            "cost_ci_upper",
            "lower_cost_policy",
            "point_fill_rate_difference_pp_"
            "alternative_minus_baseline",
            "fill_ci_lower_pp",
            "fill_ci_upper_pp",
            "higher_fill_policy"
        ]
    ].to_string(index=False)
)


print("\nSaved files:")
print(OUTPUT_FILE)
print(CHECK_FILE)


Series per demand group:
{'Overall': 500, 'Smooth': 125, 'Erratic': 125, 'Intermittent': 125, 'Lumpy': 125}
Completed paired bootstrap for Overall
Completed paired bootstrap for Smooth
Completed paired bootstrap for Erratic
Completed paired bootstrap for Intermittent
Completed paired bootstrap for Lumpy

Policy-timing paired bootstrap completed successfully.

Comparison rows: 375

Bootstrap checks:
                                             check  passed
0                         Expected comparison rows    True
1                 Exactly 2000 bootstrap resamples    True
2                            Correct series counts    True
3                Exactly five alternative policies    True
4                              Exactly five models    True
5                    Exactly three quantile levels    True
6                       Exactly five demand groups    True
7                         Finite bootstrap results    True
8  Cost interval lower bound not above upper bound    True
9  Fill 

## 6. Focused post-hoc comparison

This final cell compares ETS under `(14,7)` with XGBoost under `(14,14)` at the 0.99 target level. The comparison is exploratory because it was selected after inspecting the policy results.


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# 1. Settings
# ============================================================

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))
OUTPUT_DIR = DATA_DIR / "chapter5_results"

INPUT_FILE = (
    OUTPUT_DIR
    / "inventory_policy_timing_series_results.csv"
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "inventory_best_configuration_paired_bootstrap.csv"
)

CHECK_FILE = (
    OUTPUT_DIR
    / "inventory_best_configuration_bootstrap_checks.csv"
)

BOOTSTRAP_RESAMPLES = 2000
RANDOM_SEED = 2026
QUANTILE_LEVEL = 0.99

CONFIGURATION_A = {
    "label": "ETS_R14_L7_q0.99",
    "model": "ets",
    "review_period": 14,
    "lead_time": 7
}

CONFIGURATION_B = {
    "label": "XGBoost_R14_L14_q0.99",
    "model": "xgboost",
    "review_period": 14,
    "lead_time": 14
}


# ============================================================
# 2. Load the corrected per-series results
# ============================================================

results = pd.read_csv(INPUT_FILE)

results["id"] = results["id"].astype(str)

results["model"] = (
    results["model"]
    .astype(str)
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace(" ", "_", regex=False)
    .replace(
        {
            "tirex_2": "tirex2",
            "seasonal_naïve": "seasonal_naive"
        }
    )
)

results["demand_class"] = (
    results["demand_class"]
    .astype(str)
    .str.strip()
    .str.capitalize()
)


def select_configuration(
    data,
    configuration
):

    return data[
        (
            data["model"]
            == configuration["model"]
        )
        & (
            data["review_period"]
            == configuration["review_period"]
        )
        & (
            data["lead_time"]
            == configuration["lead_time"]
        )
        & np.isclose(
            data["quantile_level"],
            QUANTILE_LEVEL
        )
    ].copy()


configuration_a = select_configuration(
    results,
    CONFIGURATION_A
)

configuration_b = select_configuration(
    results,
    CONFIGURATION_B
)

assert len(configuration_a) == 500
assert len(configuration_b) == 500


# ============================================================
# 3. Define the demand groups
# ============================================================

group_ids = {
    "Overall": sorted(
        configuration_a["id"].unique()
    )
}

for demand_class in [
    "Smooth",
    "Erratic",
    "Intermittent",
    "Lumpy"
]:

    group_ids[demand_class] = sorted(
        configuration_a.loc[
            configuration_a["demand_class"]
            == demand_class,
            "id"
        ].unique()
    )


expected_group_sizes = {
    "Overall": 500,
    "Smooth": 125,
    "Erratic": 125,
    "Intermittent": 125,
    "Lumpy": 125
}

assert {
    group_name: len(ids)
    for group_name, ids
    in group_ids.items()
} == expected_group_sizes


# ============================================================
# 4. Create fixed paired bootstrap samples
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_indices = {
    group_name: rng.integers(
        low=0,
        high=len(ids),
        size=(
            BOOTSTRAP_RESAMPLES,
            len(ids)
        )
    )
    for group_name, ids
    in group_ids.items()
}


# ============================================================
# 5. Focused paired bootstrap
# ============================================================

comparison_rows = []

for demand_group, ids in group_ids.items():

    config_a = (
        configuration_a
        .set_index("id")
        .reindex(ids)
    )

    config_b = (
        configuration_b
        .set_index("id")
        .reindex(ids)
    )

    assert config_a[
        "total_cost"
    ].notna().all()

    assert config_b[
        "total_cost"
    ].notna().all()


    demand_a = (
        config_a["demand_units"]
        .to_numpy(dtype=float)
    )

    demand_b = (
        config_b["demand_units"]
        .to_numpy(dtype=float)
    )

    assert np.allclose(
        demand_a,
        demand_b
    )


    # A minus B:
    # Negative value = ETS R14/L7 is cheaper.
    cost_difference = (
        config_a["total_cost"]
        .to_numpy(dtype=float)
        - config_b["total_cost"]
        .to_numpy(dtype=float)
    )

    # A minus B:
    # Negative value = XGBoost R14/L14
    # has the higher fill rate.
    sales_difference = (
        config_a["sales_units"]
        .to_numpy(dtype=float)
        - config_b["sales_units"]
        .to_numpy(dtype=float)
    )

    sampled_indices = (
        bootstrap_indices[demand_group]
    )


    bootstrap_cost_difference = (
        cost_difference[
            sampled_indices
        ].sum(axis=1)
    )

    bootstrap_demand = (
        demand_a[
            sampled_indices
        ].sum(axis=1)
    )

    if np.any(bootstrap_demand <= 0):
        raise ValueError(
            "A bootstrap resample has "
            "no observed demand."
        )

    bootstrap_fill_difference_pp = (
        100
        * (
            sales_difference[
                sampled_indices
            ].sum(axis=1)
            / bootstrap_demand
        )
    )


    point_cost_difference = (
        cost_difference.sum()
    )

    point_fill_difference_pp = (
        100
        * sales_difference.sum()
        / demand_a.sum()
    )


    (
        cost_ci_lower,
        cost_ci_upper
    ) = np.quantile(
        bootstrap_cost_difference,
        [
            0.025,
            0.975
        ]
    )

    (
        fill_ci_lower,
        fill_ci_upper
    ) = np.quantile(
        bootstrap_fill_difference_pp,
        [
            0.025,
            0.975
        ]
    )


    if cost_ci_upper < 0:

        lower_cost_configuration = (
            CONFIGURATION_A["label"]
        )

    elif cost_ci_lower > 0:

        lower_cost_configuration = (
            CONFIGURATION_B["label"]
        )

    else:

        lower_cost_configuration = (
            "no_clear_difference"
        )


    if fill_ci_lower > 0:

        higher_fill_configuration = (
            CONFIGURATION_A["label"]
        )

    elif fill_ci_upper < 0:

        higher_fill_configuration = (
            CONFIGURATION_B["label"]
        )

    else:

        higher_fill_configuration = (
            "no_clear_difference"
        )


    comparison_rows.append(
        {
            "demand_group":
                demand_group,

            "series_count":
                len(ids),

            "bootstrap_resamples":
                BOOTSTRAP_RESAMPLES,

            "configuration_a":
                CONFIGURATION_A["label"],

            "configuration_b":
                CONFIGURATION_B["label"],

            "configuration_a_total_cost":
                config_a["total_cost"].sum(),

            "configuration_b_total_cost":
                config_b["total_cost"].sum(),

            "point_cost_difference_a_minus_b":
                point_cost_difference,

            "bootstrap_mean_cost_difference":
                bootstrap_cost_difference.mean(),

            "bootstrap_se_cost_difference":
                bootstrap_cost_difference.std(
                    ddof=1
                ),

            "cost_ci_lower":
                cost_ci_lower,

            "cost_ci_upper":
                cost_ci_upper,

            "cost_ci_excludes_zero":
                bool(
                    cost_ci_lower > 0
                    or cost_ci_upper < 0
                ),

            "lower_cost_configuration":
                lower_cost_configuration,

            "configuration_a_fill_rate":
                (
                    config_a["sales_units"].sum()
                    / demand_a.sum()
                ),

            "configuration_b_fill_rate":
                (
                    config_b["sales_units"].sum()
                    / demand_a.sum()
                ),

            "point_fill_difference_pp_a_minus_b":
                point_fill_difference_pp,

            "bootstrap_mean_fill_difference_pp":
                (
                    bootstrap_fill_difference_pp
                    .mean()
                ),

            "bootstrap_se_fill_difference_pp":
                (
                    bootstrap_fill_difference_pp
                    .std(ddof=1)
                ),

            "fill_ci_lower_pp":
                fill_ci_lower,

            "fill_ci_upper_pp":
                fill_ci_upper,

            "fill_ci_excludes_zero":
                bool(
                    fill_ci_lower > 0
                    or fill_ci_upper < 0
                ),

            "higher_fill_configuration":
                higher_fill_configuration,

            "holding_cost_difference_a_minus_b":
                (
                    config_a[
                        "holding_cost"
                    ].sum()
                    - config_b[
                        "holding_cost"
                    ].sum()
                ),

            "ordering_cost_difference_a_minus_b":
                (
                    config_a[
                        "ordering_cost"
                    ].sum()
                    - config_b[
                        "ordering_cost"
                    ].sum()
                ),

            "lost_sales_cost_difference_a_minus_b":
                (
                    config_a[
                        "lost_sales_cost"
                    ].sum()
                    - config_b[
                        "lost_sales_cost"
                    ].sum()
                ),

            "average_inventory_difference_a_minus_b":
                (
                    config_a[
                        "average_inventory"
                    ].mean()
                    - config_b[
                        "average_inventory"
                    ].mean()
                ),

            "order_count_difference_a_minus_b":
                (
                    config_a[
                        "order_count"
                    ].sum()
                    - config_b[
                        "order_count"
                    ].sum()
                )
        }
    )

    print(
        "Completed focused bootstrap for",
        demand_group
    )


focused_results = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 6. Implementation checks
# ============================================================

bootstrap_checks = pd.DataFrame(
    {
        "check": [
            "Exactly five comparison rows",
            "Exactly 2000 bootstrap resamples",
            "Correct series counts",
            "Finite numerical results",
            "Cost interval lower bound not above upper bound",
            "Fill interval lower bound not above upper bound"
        ],

        "passed": [
            len(focused_results) == 5,

            focused_results[
                "bootstrap_resamples"
            ].eq(
                BOOTSTRAP_RESAMPLES
            ).all(),

            focused_results.apply(
                lambda row:
                    row["series_count"]
                    == expected_group_sizes[
                        row["demand_group"]
                    ],
                axis=1
            ).all(),

            np.isfinite(
                focused_results.select_dtypes(
                    include=[np.number]
                )
            ).all().all(),

            (
                focused_results[
                    "cost_ci_lower"
                ]
                <= focused_results[
                    "cost_ci_upper"
                ]
            ).all(),

            (
                focused_results[
                    "fill_ci_lower_pp"
                ]
                <= focused_results[
                    "fill_ci_upper_pp"
                ]
            ).all()
        ]
    }
)

if not bootstrap_checks["passed"].all():

    failed_checks = bootstrap_checks.loc[
        ~bootstrap_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Focused bootstrap checks failed: "
        f"{failed_checks}"
    )


# ============================================================
# 7. Save and display results
# ============================================================

focused_results.to_csv(
    OUTPUT_FILE,
    index=False
)

bootstrap_checks.to_csv(
    CHECK_FILE,
    index=False
)


print(
    "\nFocused paired bootstrap "
    "completed successfully.\n"
)

print("Checks:")
print(bootstrap_checks)

print(
    "\nETS R14/L7 minus XGBoost "
    "R14/L14 at q=0.99:"
)

print(
    focused_results[
        [
            "demand_group",
            "series_count",
            "point_cost_difference_a_minus_b",
            "cost_ci_lower",
            "cost_ci_upper",
            "lower_cost_configuration",
            "point_fill_difference_pp_a_minus_b",
            "fill_ci_lower_pp",
            "fill_ci_upper_pp",
            "higher_fill_configuration"
        ]
    ].to_string(index=False)
)

print("\nSaved files:")
print(OUTPUT_FILE)
print(CHECK_FILE)


Completed focused bootstrap for Overall
Completed focused bootstrap for Smooth
Completed focused bootstrap for Erratic
Completed focused bootstrap for Intermittent
Completed focused bootstrap for Lumpy

Focused paired bootstrap completed successfully.

Checks:
                                             check  passed
0                     Exactly five comparison rows    True
1                 Exactly 2000 bootstrap resamples    True
2                            Correct series counts    True
3                         Finite numerical results    True
4  Cost interval lower bound not above upper bound    True
5  Fill interval lower bound not above upper bound    True

ETS R14/L7 minus XGBoost R14/L14 at q=0.99:
demand_group  series_count  point_cost_difference_a_minus_b  cost_ci_lower  cost_ci_upper lower_cost_configuration  point_fill_difference_pp_a_minus_b  fill_ci_lower_pp  fill_ci_upper_pp higher_fill_configuration
     Overall           500                        -4.359123     -19.